In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [8]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).text

    return {'joke': response}

def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).text

    return {'explanation': response}

In [9]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [10]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "My wife told me to stop impersonating a flamingo. \n\nI had to put my foot down. \n\n...Which is also what I do when the delivery guy forgets to slice my pizza, but I don't want to make a *major* issue out of it.",
 'explanation': 'This joke works by combining a classic "dad joke" pun with a layered pizza-related pun. Here is the step-by-step breakdown:\n\n1. **The Setup:** *“My wife told me to stop impersonating a flamingo.”* \n   * Flamingos are famous for standing on one leg. \n\n2. **The First Punchline:** *“I had to put my foot down.”* \n   * **Literal meaning:** To stand on one leg, you physically have to put your other foot down on the ground (thus stopping the flamingo impersonation).\n   * **Idiomatic meaning:** "To put your foot down" is a common phrase that means to be firm, assert your authority, or draw a line (e.g., *“My wife told me to stop, but I put my foot down and kept doing it”*). \n\n3. **The Twist / Tagline:** *“...Which is also what I

In [11]:

workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "My wife told me to stop impersonating a flamingo. \n\nI had to put my foot down. \n\n...Which is also what I do when the delivery guy forgets to slice my pizza, but I don't want to make a *major* issue out of it.", 'explanation': 'This joke works by combining a classic "dad joke" pun with a layered pizza-related pun. Here is the step-by-step breakdown:\n\n1. **The Setup:** *“My wife told me to stop impersonating a flamingo.”* \n   * Flamingos are famous for standing on one leg. \n\n2. **The First Punchline:** *“I had to put my foot down.”* \n   * **Literal meaning:** To stand on one leg, you physically have to put your other foot down on the ground (thus stopping the flamingo impersonation).\n   * **Idiomatic meaning:** "To put your foot down" is a common phrase that means to be firm, assert your authority, or draw a line (e.g., *“My wife told me to stop, but I put my foot down and kept doing it”*). \n\n3. **The Twist / Tagline:** *“...W

In [12]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "My wife told me to stop impersonating a flamingo. \n\nI had to put my foot down. \n\n...Which is also what I do when the delivery guy forgets to slice my pizza, but I don't want to make a *major* issue out of it.", 'explanation': 'This joke works by combining a classic "dad joke" pun with a layered pizza-related pun. Here is the step-by-step breakdown:\n\n1. **The Setup:** *“My wife told me to stop impersonating a flamingo.”* \n   * Flamingos are famous for standing on one leg. \n\n2. **The First Punchline:** *“I had to put my foot down.”* \n   * **Literal meaning:** To stand on one leg, you physically have to put your other foot down on the ground (thus stopping the flamingo impersonation).\n   * **Idiomatic meaning:** "To put your foot down" is a common phrase that means to be firm, assert your authority, or draw a line (e.g., *“My wife told me to stop, but I put my foot down and kept doing it”*). \n\n3. **The Twist / Tagline:** *“...

In [13]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1ba5ab-68b0-64fb-8000-403674379d3b"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1ba5ab-68b0-64fb-8000-403674379d3b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-27T10:03:39.758274+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ba5ab-68a8-6279-bfff-e12fdbc16f35'}}, tasks=(PregelTask(id='fad54988-a493-5b92-29e0-3446da65ab69', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "My wife told me to stop impersonating a flamingo. \n\nI had to put my foot down. \n\n...Which is also what I do when the delivery guy forgets to slice my pizza, but I don't want to make a *major* issue out of it."}),), interrupts=())

In [14]:
workflow.invoke(None,{"configurable": {"thread_id": "1", "checkpoint_id": "1f1ba5ab-68b0-64fb-8000-403674379d3b"}})

{'topic': 'pizza',
 'joke': 'Why did the hipster burn the roof of his mouth?\n\nBecause he ate the pizza before it was cool.',
 'explanation': 'Here is the explanation for the joke:\n\n**The Joke:**\n*“Why did the hipster burn the roof of his mouth? Because he ate the pizza before it was cool.”*\n\n**The Explanation:**\nThis joke relies on a **pun** (a play on words) based on a common cultural stereotype about hipsters. \n\n1. **The Stereotype:** Hipsters are often stereotyped as trying very hard to be trendy, unique, and ahead of the curve. A classic cliché associated with them is saying, "I liked that band *before* it was cool" (meaning they discovered it before it became mainstream). \n2. **The Literal Meaning:** In the context of food, "cool" literally means having a lower temperature. Freshly baked pizza is piping hot, so you usually have to wait for it to cool down before eating it to avoid burning your mouth.\n\n**The Punchline:** \nBy combining these two ideas, the joke creates

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the hipster burn the roof of his mouth?\n\nBecause he ate the pizza before it was cool.', 'explanation': 'Here is the explanation for the joke:\n\n**The Joke:**\n*“Why did the hipster burn the roof of his mouth? Because he ate the pizza before it was cool.”*\n\n**The Explanation:**\nThis joke relies on a **pun** (a play on words) based on a common cultural stereotype about hipsters. \n\n1. **The Stereotype:** Hipsters are often stereotyped as trying very hard to be trendy, unique, and ahead of the curve. A classic cliché associated with them is saying, "I liked that band *before* it was cool" (meaning they discovered it before it became mainstream). \n2. **The Literal Meaning:** In the context of food, "cool" literally means having a lower temperature. Freshly baked pizza is piping hot, so you usually have to wait for it to cool down before eating it to avoid burning your mouth.\n\n**The Punchline:** \nBy combining these two ide

In [16]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1ba5ab-68b0-64fb-8000-403674379d3b", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1ba5cb-63f0-675c-8001-2c7928f44513'}}

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ba5cb-63f0-675c-8001-2c7928f44513'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-27T10:17:58.253715+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ba5ab-68b0-64fb-8000-403674379d3b'}}, tasks=(PregelTask(id='e5dda859-2746-f4fb-503c-41bbc3a06844', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the hipster burn the roof of his mouth?\n\nBecause he ate the pizza before it was cool.', 'explanation': 'Here is the explanation for the joke:\n\n**The Joke:**\n*“Why did the hipster burn the roof of his mouth? Because he ate the pizza before it was cool.”*\n\n**The Explanation:**\nThis joke relies on a **pun** (a play 

In [18]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1ba5cb-63f0-675c-8001-2c7928f44513"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it felt like everyone was just trying to get inside its head, and honestly, it was at its breaking point. \n\nPlus, it had too many mixed feelings bottled up inside!',
 'explanation': 'Here is an explanation of the joke, broken down by its punchlines:\n\n**The Setup:**\n"Why did the samosa go to therapy?" \nThis sets up a classic "therapy" joke scenario, usually reserved for someone who is stressed out, overwhelmed, and needs professional help. \n\n**Layer 1: "It felt like everyone was just trying to get inside its head..."**\n*   **Literal meaning:** A therapist tries to "get inside a patient\'s head" to understand their thoughts and feelings. \n*   **The Samosa twist:** A samosa is a stuffed pastry. Literally, the only reason anyone interacts with a samosa is to bite into it and "get inside" its shell to eat the filling. \n\n**Layer 2: "...and honestly, it was at its breaking point."**\n*   **Literal meaning:*

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it felt like everyone was just trying to get inside its head, and honestly, it was at its breaking point. \n\nPlus, it had too many mixed feelings bottled up inside!', 'explanation': 'Here is an explanation of the joke, broken down by its punchlines:\n\n**The Setup:**\n"Why did the samosa go to therapy?" \nThis sets up a classic "therapy" joke scenario, usually reserved for someone who is stressed out, overwhelmed, and needs professional help. \n\n**Layer 1: "It felt like everyone was just trying to get inside its head..."**\n*   **Literal meaning:** A therapist tries to "get inside a patient\'s head" to understand their thoughts and feelings. \n*   **The Samosa twist:** A samosa is a stuffed pastry. Literally, the only reason anyone interacts with a samosa is to bite into it and "get inside" its shell to eat the filling. \n\n**Layer 2: "...and honestly, it was at its breaking point."**\n*  